# Test Case Generation Agent — A2A client

Calls the running A2A server for `test_case_generation_agent` and prints the generated test cases.

Start the server first, from the `agents/` directory:

```
uvicorn test_case_agent.a2a_server:a2a_app --host 0.0.0.0 --port 8001
```

In [9]:
import json
import uuid

import requests

A2A_BASE_URL = "http://localhost:8001"

In [10]:
def get_agent_card(base_url: str = A2A_BASE_URL) -> dict:
    """Fetch the agent card to confirm the server is up and see its capabilities."""
    resp = requests.get(f"{base_url}/.well-known/agent-card.json", timeout=10)
    resp.raise_for_status()
    return resp.json()


get_agent_card()

{'name': 'test_case_generation_agent',
 'description': 'Analyzes Jira requirements and existing test cases, identifies test coverage gaps, and generates only new non-duplicate test cases.',
 'supportedInterfaces': [{'url': 'http://localhost:8001',
   'protocolBinding': 'JSONRPC',
   'protocolVersion': '1.0'}],
 'version': '0.0.1',
 'capabilities': {'streaming': True},
 'defaultInputModes': ['text/plain'],
 'defaultOutputModes': ['text/plain'],
 'skills': [{'id': 'test_case_generation_agent',
   'name': 'model',
   'description': 'Analyzes Jira requirements and existing test cases, identifies test coverage gaps, and generates only new non-duplicate test cases.',
   'tags': ['llm']}]}

In [11]:
def call_test_case_agent(jira_context: dict, base_url: str = A2A_BASE_URL) -> dict:
    """Send a Jira context to the A2A-exposed test_case_generation_agent and
    return the parsed TestCaseGenerationResult dict.

    Requires the A2A-Version: 1.0 header — without it the request defaults
    to protocol 0.3, which this ADK server rejects.
    """
    envelope = {
        "jsonrpc": "2.0",
        "id": str(uuid.uuid4()),
        "method": "SendMessage",
        "params": {
            "message": {
                "messageId": str(uuid.uuid4()),
                "role": "ROLE_USER",
                "parts": [
                    {"text": json.dumps(jira_context, ensure_ascii=False)}
                ],
            }
        },
    }

    resp = requests.post(
        f"{base_url}/",
        headers={
            "Content-Type": "application/json",
            "A2A-Version": "1.0",
        },
        json=envelope,
        timeout=120,
    )
    resp.raise_for_status()
    body = resp.json()

    if "error" in body:
        error = body["error"]
        raise RuntimeError(f"A2A error {error['code']}: {error['message']}")

    task = body["result"]["task"]
    state = task["status"]["state"]

    if state == "TASK_STATE_FAILED":
        # On failure the error text lands in status.message.
        failure_text = task["status"]["message"]["parts"][0]["text"]
        raise RuntimeError(f"Agent task failed: {failure_text}")

    # On success the actual TestCaseGenerationResult JSON is an artifact,
    # not status.message (status carries no message at all on success).
    result_text = task["artifacts"][0]["parts"][0]["text"]
    return json.loads(result_text)

In [12]:
sample_jira_context = {
    "epic_key": "SCRUM-7",
    "story_key": "SCRUM-24",
    "summary": "Display remaining OTP attempts",
    "description": (
        "As a banking user, I want to see how many OTP attempts are "
        "remaining so that I understand when my transaction may be "
        "blocked. After every incorrect OTP entry, the system should "
        "display the remaining retry count. The message should be clear "
        "and updated immediately. When no attempts remain, the system "
        "should show that the OTP verification has failed and stop the "
        "transaction."
    ),
    "priority": "High",
    "acceptance_criteria": [
        {"id": "AC-1", "text": "System displays the remaining OTP retry count after every incorrect OTP entry."},
        {"id": "AC-2", "text": "Remaining retry count is updated immediately after each failed OTP attempt."},
        {"id": "AC-3", "text": "Message clearly informs the user how many OTP attempts are remaining."},
        {"id": "AC-4", "text": "When no attempts remain, system shows an OTP verification failed message."},
        {"id": "AC-5", "text": "Transaction is stopped when OTP attempts are exhausted."},
        {"id": "AC-6", "text": "User cannot continue the transaction after OTP verification has failed."},
    ],
    "existing_test_cases": [
        {
            "test_case_id": "TC-101",
            "title": "Existing Jira test case",
            "preconditions": [],
            "test_data": {},
            "steps": [],
            "expected_result": "",
            "mapped_acceptance_criteria": [],
        }
    ],
    "linked_defects": [],
    "business_rules": [],
    "related_issues": [],
}

result = call_test_case_agent(sample_jira_context)
result

{'story_key': 'SCRUM-24',
 'requirement_summary': 'The system must display and accurately update the remaining OTP attempts to banking users after incorrect entries, clearly communicating when attempts are exhausted and ensuring transactions are halted if verification fails.',
 'coverage_analysis': [{'acceptance_criterion_id': 'AC-1',
   'status': 'NOT_COVERED',
   'existing_test_case_ids': [],
   'coverage_gap': 'No test case validates that the system displays the remaining OTP retry count after each incorrect OTP entry.'},
  {'acceptance_criterion_id': 'AC-2',
   'status': 'NOT_COVERED',
   'existing_test_case_ids': [],
   'coverage_gap': 'No test case checks that the retry count is updated immediately after each failed OTP attempt.'},
  {'acceptance_criterion_id': 'AC-3',
   'status': 'NOT_COVERED',
   'existing_test_case_ids': [],
   'coverage_gap': 'There is no test case confirming that the message clearly informs the user how many OTP attempts are remaining.'},
  {'acceptance_cri

In [13]:
# Pretty-print the full result, or drill into a specific part, e.g.:
print(json.dumps(result, indent=2))

# result["coverage_analysis"]
# result["generated_test_cases"]

{
  "story_key": "SCRUM-24",
  "requirement_summary": "The system must display and accurately update the remaining OTP attempts to banking users after incorrect entries, clearly communicating when attempts are exhausted and ensuring transactions are halted if verification fails.",
  "coverage_analysis": [
    {
      "acceptance_criterion_id": "AC-1",
      "status": "NOT_COVERED",
      "existing_test_case_ids": [],
      "coverage_gap": "No test case validates that the system displays the remaining OTP retry count after each incorrect OTP entry."
    },
    {
      "acceptance_criterion_id": "AC-2",
      "status": "NOT_COVERED",
      "existing_test_case_ids": [],
      "coverage_gap": "No test case checks that the retry count is updated immediately after each failed OTP attempt."
    },
    {
      "acceptance_criterion_id": "AC-3",
      "status": "NOT_COVERED",
      "existing_test_case_ids": [],
      "coverage_gap": "There is no test case confirming that the message clearly inf